In [3]:
!pip install openai requests

In [12]:
import os
import json
import requests
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# =====================================================================
# 1. Tool Definitions (Standard Python Functions)
# =====================================================================

def check_server_status(url: str) -> str:
    if not url.startswith("http"):
        url = f"https://{url}"
    try:
        response = requests.get(url, timeout=5)
        return json.dumps({"status": "Online", "status_code": response.status_code, "latency_ms": int(response.elapsed.total_seconds() * 1000)})
    except Exception as e:
        return json.dumps({"status": "Offline", "error": str(e)})

def get_exchange_rate(base_currency: str, target_currency: str) -> str:
    base = base_currency.upper()
    target = target_currency.upper()
    try:
        url = f"https://open.er-api.com/v6/latest/{base}"
        res = requests.get(url, timeout=5).json()
        if "rates" in res and target in res["rates"]:
            return json.dumps({"base": base, "target": target, "rate": res["rates"][target]})
        return json.dumps({"error": f"Currency {target} not found."})
    except Exception as e:
        return json.dumps({"error": str(e)})

def inspect_github_repo(owner: str, repo: str) -> str:
    try:
        url = f"https://api.github.com/repos/{owner}/{repo}"
        headers = {"User-Agent": "Kaggle-Agent"}
        data = requests.get(url, headers=headers, timeout=5).json()
        return json.dumps({
            "name": data.get("full_name"),
            "stars": data.get("stargazers_count"),
            "forks": data.get("forks_count"),
            "open_issues": data.get("open_issues_count")
        })
    except Exception as e:
        return json.dumps({"error": str(e)})

# =====================================================================
# 2. Schema Mapping (Connecting Code to the SLM)
# =====================================================================

available_functions = {
    "check_server_status": check_server_status,
    "get_exchange_rate": get_exchange_rate,
    "inspect_github_repo": inspect_github_repo
}

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "check_server_status",
            "description": "Ping a website URL to verify if it is reachable.",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string", "description": "The domain name or web address, e.g., google.com"}},
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Convert values or find rates between different standard currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_currency": {"type": "string", "description": "Starting currency, e.g. USD"},
                    "target_currency": {"type": "string", "description": "Destination currency, e.g. INR"}
                },
                "required": ["base_currency", "target_currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "inspect_github_repo",
            "description": "Get stars and forks for a public github repo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "GitHub username/organization"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    }
]

# =====================================================================
# 3. Execution Infrastructure Loop
# =====================================================================

def run_kaggle_agent():
    try:
        # Securely fetch the API key from Kaggle Secrets
        user_secrets = UserSecretsClient()
        groq_key = user_secrets.get_secret("GROQ_API_KEY")
    except Exception as e:
        print("⚠️ Error: Could not find GROQ_API_KEY in Kaggle Secrets. Please add it first!")
        return

    # Initialize the client using the secure key variable
    client = OpenAI(
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key
    )
    
    model_name = "meta-llama/llama-4-scout-17b-16e-instruct"

    messages = [
        {"role": "system", "content": "You are a developer utility assistant. Use your tools selectively."}
    ]

    print("⚡ Agent Operational on Kaggle Infrastructure.")
    print("Type 'exit' to stop the loop.\n")

    while True:
        # This will open a text box directly inside your Kaggle notebook cell!
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit']:
            print("Session closed.")
            break

        messages.append({"role": "user", "content": user_input})

        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            messages.append(response_message)
            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"\n[Agent Routing: Triggering '{function_name}']")
                
                if function_name in available_functions:
                    function_to_call = available_functions[function_name]
                    function_response = function_to_call(**function_args)
                else:
                    function_response = json.dumps({"error": "Unknown tool mapping"})
                
                print(f"[Tool Raw JSON: {function_response}]")
                
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                })

            second_response = client.chat.completions.create(
                model=model_name,
                messages=messages
            )
            final_reply = second_response.choices[0].message.content
            print(f"\nAgent: {final_reply}\n")
            messages.append({"role": "assistant", "content": final_reply})
        else:
            reply = response_message.content
            print(f"\nAgent: {reply}\n")
            messages.append({"role": "assistant", "content": reply})

# Trigger the loop
run_kaggle_agent()

⚡ Agent Operational on Kaggle Infrastructure.
Type 'exit' to stop the loop.



You:  hi



Agent: Hi! How can I assist you today?



You:  Check if wireshark.org is online. Then, inspect the wireshark/wireshark repository on GitHub to see how many open issues it has. Finally, if I want to buy a 45 EUR network traffic analysis textbook, how much is that in INR?



[Agent Routing: Triggering 'check_server_status']
[Tool Raw JSON: {"status": "Online", "status_code": 200, "latency_ms": 125}]

[Agent Routing: Triggering 'inspect_github_repo']
[Tool Raw JSON: {"name": "wireshark/wireshark", "stars": 9445, "forks": 2152, "open_issues": 2}]

[Agent Routing: Triggering 'get_exchange_rate']
[Tool Raw JSON: {"base": "EUR", "target": "INR", "rate": 110.352997}]

Agent: I've checked on the status of wireshark.org and it seems to be online.

The wireshark/wireshark repository on GitHub currently has 2 open issues.

As for the textbook, if it costs 45 EUR, and assuming an exchange rate of 110.35 INR per EUR, that would be approximately 4965.98 INR.



...


Agent: Skoda is a Czech automobile manufacturer, and its origin dates back to 1895. The company was founded by Václav Laurin and Václav Klement in Mladá Boleslav, Bohemia (now part of the Czech Republic). Initially, the company was called Laurin & Klement and produced bicycles. In 1899, they began manufacturing motorcycles, and in 1906, they started producing cars. The company was nationalized in 1945 and became a state-owned enterprise. After the fall of communism, Skoda was privatized and became a subsidiary of the German automobile manufacturer Volkswagen Group in 1991. Today, Skoda is a well-known brand that produces a range of cars, including sedans, SUVs, and hatchbacks.



You:  thanks 



Agent: You're welcome! It was a pleasure assisting you with checking the status of wireshark.org, inspecting the wireshark repository on GitHub, converting the EUR to INR exchange rate, and providing information about Skoda's origin. If you have any more questions or need help with anything else, feel free to ask!



You:  exit


Session closed.


In [13]:
pip install fastapi uvicorn openai requests

Note: you may need to restart the kernel to use updated packages.


In [17]:
!pip install nest_asyncio

In [19]:
import os
import json
import requests
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

app = FastAPI(title="Agentic Webhook Backend")

# Securely fetch the API key from Kaggle Secrets
try:
    groq_key = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception as e:
    raise RuntimeError("⚠️ Could not find GROQ_API_KEY in Kaggle Secrets. Make sure it is attached to the notebook!")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key
)

model_name = "llama-3.3-70b-versatile"

# =====================================================================
# Python Tools definitions
# =====================================================================
def check_server_status(url: str) -> str:
    if not url.startswith("http"):
        url = f"https://{url}"
    try:
        response = requests.get(url, timeout=5)
        return json.dumps({
            "status": "Online", 
            "status_code": response.status_code, 
            "latency_ms": int(response.elapsed.total_seconds() * 1000)
        })
    except Exception as e:
        return json.dumps({"status": "Offline", "error": str(e)})

def get_exchange_rate(base_currency: str, target_currency: str) -> str:
    try:
        url = f"https://open.er-api.com/v6/latest/{base_currency.upper()}"
        res = requests.get(url, timeout=5).json()
        target = target_currency.upper()
        if "rates" in res and target in res["rates"]:
            return json.dumps({
                "base": base_currency.upper(), 
                "target": target, 
                "rate": res["rates"][target]
            })
        return json.dumps({"error": f"Currency {target} not found."})
    except Exception as e:
        return json.dumps({"error": str(e)})

def inspect_github_repo(owner: str, repo: str) -> str:
    try:
        url = f"https://api.github.com/repos/{owner}/{repo}"
        headers = {"User-Agent": "FastAPI-Agent"}
        data = requests.get(url, headers=headers, timeout=5).json()
        return json.dumps({
            "name": data.get("full_name"),
            "stars": data.get("stargazers_count"),
            "forks": data.get("forks_count"),
            "open_issues": data.get("open_issues_count")
        })
    except Exception as e:
        return json.dumps({"error": str(e)})

available_functions = {
    "check_server_status": check_server_status,
    "get_exchange_rate": get_exchange_rate,
    "inspect_github_repo": inspect_github_repo
}

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "check_server_status",
            "description": "Ping a website URL to verify if it is reachable.",
            "parameters": {
                "type": "object",
                "properties": {"url": {"type": "string", "description": "The domain name, e.g., google.com"}},
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_exchange_rate",
            "description": "Convert values or find rates between different standard currencies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_currency": {"type": "string", "description": "Starting currency, e.g. USD"},
                    "target_currency": {"type": "string", "description": "Destination currency, e.g. INR"}
                },
                "required": ["base_currency", "target_currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "inspect_github_repo",
            "description": "Get stars and forks for a public github repo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "GitHub username"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    }
]

# =====================================================================
# API Request Schema for incoming Typebot messages
# =====================================================================
class ChatRequest(BaseModel):
    message: str

@app.post("/agent/chat")
async def agent_chat_endpoint(payload: ChatRequest):
    messages = [
        {"role": "system", "content": "You are a professional full-stack system utility assistant. Answer clearly and use tools when required."},
        {"role": "user", "content": payload.message}
    ]

    try:
        # First LLM pass
        response = client.chat.completions.create(
            model=model_name,
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            # Cleanly format the assistant's tool-call response
            messages.append({
                "role": "assistant",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in tool_calls
                ]
            })

            # Execute tools
            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"Triggering: {function_name} with {function_args}")
                
                if function_name in available_functions:
                    function_to_call = available_functions[function_name]
                    function_response = function_to_call(**function_args)
                else:
                    function_response = json.dumps({"error": "Unknown tool"})
                
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                })

            # Final synthesis
            second_response = client.chat.completions.create(
                model=model_name,
                messages=messages
            )
            return {"reply": second_response.choices[0].message.content}
        else:
            return {"reply": response_message.content}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# =====================================================================
# Notebook Execution Entrypoint (Kaggle/Jupyter Native)
# =====================================================================
if __name__ == "__main__":
    import uvicorn
    
    print("🚀 FastAPI Agent Server starting on port 8000...")
    
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    
    await server.serve()

/usr/local/lib/python3.12/dist-packages/IPython/core/compilerop.py:101: RuntimeWarning: coroutine 'Server.serve' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)


🚀 FastAPI Agent Server starting on port 8000...


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [58]
